In [1]:
import numpy as np
import torch
import gc
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset
from tsai.models.InceptionTime import InceptionTime
warnings.filterwarnings('ignore')

print("✅ 导入完成")

✅ 导入完成


In [2]:
X = np.load('/root/X_interpolated.npy')
y = np.load('/root/y_interpolated.npy')

print(f"X 形状: {X.shape}")
print(f"y 分布: 0={sum(y==0)}, 1={sum(y==1)}")
print(f"X 中有 NaN: {np.isnan(X).any()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

X 形状: (14662, 4096, 23)
y 分布: 0=7595, 1=7067
X 中有 NaN: False
使用设备: cuda


In [3]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_labels.extend(y_batch.numpy())
    return np.array(all_preds), np.array(all_probs), np.array(all_labels)

print("✅ 训练函数定义完成")

✅ 训练函数定义完成


In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracies, f1_scores, auc_scores = [], [], []

print("\n" + "="*60)
print("InceptionTime 5折交叉验证（完整数据，跑满50轮）")
print("="*60)

fold = 1
for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    print(f"\n--- Fold {fold} ---")
    print(f"  训练集: {len(train_idx):,} 样本")
    print(f"  验证集: {len(val_idx):,} 样本")
    
    X_train_t = torch.tensor(np.transpose(X_train, (0, 2, 1)), dtype=torch.float32)
    X_val_t = torch.tensor(np.transpose(X_val, (0, 2, 1)), dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_val_t = torch.tensor(y_val, dtype=torch.long)
    
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    
    model = InceptionTime(c_in=23, c_out=2, seq_len=4096).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = torch.nn.CrossEntropyLoss()
    
    # 🔥 删除了 Early Stopping，固定跑 50 轮
    for epoch in range(50):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        if epoch % 10 == 0:
            print(f"  Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
    
    y_pred, y_prob, _ = evaluate(model, val_loader, device)
    
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)
    
    print(f"  准确率: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")
    fold += 1
    
    del model, X_train_t, X_val_t, y_train_t, y_val_t
    gc.collect()

print("\n" + "="*60)
print("📊 InceptionTime 最终结果（跑满50轮）")
print("="*60)
print(f"  准确率: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"  F1分数: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  AUC:    {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*60)


InceptionTime 5折交叉验证（完整数据，跑满50轮）

--- Fold 1 ---
  训练集: 11,729 样本
  验证集: 2,933 样本
  Epoch 0: train_loss=0.6994, val_loss=0.7084
  Epoch 10: train_loss=0.6827, val_loss=0.7090
  Epoch 20: train_loss=0.6418, val_loss=0.6536
  Epoch 30: train_loss=0.6201, val_loss=0.7205
  Epoch 40: train_loss=0.6090, val_loss=0.7958
  准确率: 0.6140, F1: 0.6538, AUC: 0.7001

--- Fold 2 ---
  训练集: 11,729 样本
  验证集: 2,933 样本
  Epoch 0: train_loss=0.7019, val_loss=0.7087
  Epoch 10: train_loss=0.6812, val_loss=0.7206
  Epoch 20: train_loss=0.6433, val_loss=0.6668
  Epoch 30: train_loss=0.6289, val_loss=0.6300
  Epoch 40: train_loss=0.6181, val_loss=0.7235
  准确率: 0.5401, F1: 0.4183, AUC: 0.5541

--- Fold 3 ---
  训练集: 11,730 样本
  验证集: 2,932 样本
  Epoch 0: train_loss=0.6969, val_loss=0.6944
  Epoch 10: train_loss=0.6874, val_loss=0.6999
  Epoch 20: train_loss=0.6515, val_loss=0.6525
  Epoch 30: train_loss=0.6270, val_loss=0.6583
  Epoch 40: train_loss=0.6131, val_loss=0.6071
  准确率: 0.6412, F1: 0.6382, AUC: 0.7222
